In [1]:
import ee
import time
 
# Initialize Earth Engine
ee.Initialize()
 
# Define Ketapang Regency boundary
ketapang = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(
    ee.Filter.eq('ADM2_NAME', 'Ketapang')
)
 
# Load palm oil collection
collection = ee.ImageCollection('projects/forestdatapartnership/assets/palm/model_2025a')
 
# Get 2020 and 2023 mosaics
p2020 = collection.filterDate('2020-01-01', '2020-12-31').mosaic()
p2023 = collection.filterDate('2023-01-01', '2023-12-31').mosaic()
 
# Apply threshold: only keep pixels >= 0.5 (50% probability)
threshold = 0.7
p2020_threshold = p2020.updateMask(p2020.gte(threshold))
p2023_threshold = p2023.updateMask(p2023.gte(threshold))
 
# Clip to Ketapang geometry
p2020_kt = p2020_threshold.clip(ketapang.geometry())
p2023_kt = p2023_threshold.clip(ketapang.geometry())

In [2]:
# Create and START task 1 - Palm 2020
task1 = ee.batch.Export.image.toDrive(
    image=p2020_kt,
    description='ketapang_palm_2020_threshold70',
    folder='GEE_Exports',
    fileNamePrefix='ketapang_palm_2020',
    region=ketapang.geometry(),
    scale=10,
    maxPixels=1e13,
    crs='EPSG:4326'
)
print("Starting task 1...")
task1.start()
time.sleep(2)  # Give it a moment to register
print("Task 1 status:", task1.status())
 
# Create and START task 2 - Palm 2023
task2 = ee.batch.Export.image.toDrive(
    image=p2023_kt,
    description='ketapang_palm_2023_threshold70',
    folder='GEE_Exports',
    fileNamePrefix='ketapang_palm_2023',
    region=ketapang.geometry(),
    scale=10,
    maxPixels=1e13,
    crs='EPSG:4326'
)
print("\nStarting task 2...")
task2.start()
time.sleep(2)
print("Task 2 status:", task2.status())
 
# List all recent tasks to verify
print("\n--- Recent Tasks ---")
tasks = ee.batch.Task.list()
for task in tasks[:5]:
    status = task.status()
    print(f"{status['description']}: {status['state']}")


Starting task 1...
Task 1 status: {'state': 'READY', 'description': 'ketapang_palm_2020_threshold70', 'priority': 100, 'creation_timestamp_ms': 1759919056862, 'update_timestamp_ms': 1759919056862, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': '2JQ5HE7QZHEQU2K5T5I6YLUS', 'name': 'projects/351618454440/operations/2JQ5HE7QZHEQU2K5T5I6YLUS'}

Starting task 2...
Task 2 status: {'state': 'READY', 'description': 'ketapang_palm_2023_threshold70', 'priority': 100, 'creation_timestamp_ms': 1759919059948, 'update_timestamp_ms': 1759919059948, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'ILMBU4SMUAKT5OKHQAYQJ47O', 'name': 'projects/351618454440/operations/ILMBU4SMUAKT5OKHQAYQJ47O'}

--- Recent Tasks ---
ketapang_palm_2023_threshold70: READY
ketapang_palm_2020_threshold70: READY
west_kalimantan_forest_loss_year_2000_2024: CANCELLED
west_kalimantan_forest_loss_2000_2024: CANCELLED
west_kalimantan_forest_loss_year_2000_2024: COMPLETED
